# 새 라벨 기반 RF-DETR-S 20epoch 학습·중단 복구

이 노트북은 2026-10-06 승인된 자료와 설정만 복원합니다. 기존 웹 노트북의 과거 Faster/D-FINE 셀은 실행하지 않습니다.

1. 브라우저만 끊긴 경우에는 기존 런타임에 재연결합니다. 서버 학습이 실행 중이면 그대로 둡니다.
2. 서버가 종료된 경우에는 세션 관리에서 기존 서버가 실행 중이 아님을 먼저 확인합니다. 무료 T4를 연결하고 복원 셀을 실행합니다.
3. 복원 후 재개 셀을 실행합니다. 정상 완료 epoch의 전체 학습 상태를 복원합니다. 미완료 epoch는 다시 계산합니다.
4. 무료 GPU 할당 제한이 있으면 할당이 가능해질 때까지 기다립니다. 다른 계정·유료 구매·연결 유지 클릭은 사용하지 않습니다.

학습 자료 12,161장 / 검증 107장 / 테스트 97장. 테스트는 검증에서 저장본과 임계값을 선택한 뒤 실행합니다.

Drive 저장소: `MyDrive/xray_direct_training_20261006`.
실제 T4·Drive에서 1epoch 저장→프로세스 종료→2epoch 재개 시험을 통과했습니다. 서버 강제 종료 시험은 하지 않았습니다.


In [ ]:
# 새 무료 T4 런타임에만 실행: 같은 Drive 자료와 고정 환경 복원
import hashlib, json, subprocess, sys, shutil, zipfile, os
os.environ['MPLBACKEND']='Agg'
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
STORE=Path('/content/drive/MyDrive/xray_direct_training_20261006')
ROOT=Path('/content/xray_direct_project')
PY='/content/xray_direct_env312/bin/python'
def digest(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)
    return h.hexdigest()
assert STORE.is_dir() and (STORE/'environment.lock.txt').is_file(), '기존 Drive 자료 연결 필요'
active=[x for x in subprocess.check_output(['ps','-eo','pid,args'],text=True).splitlines() if 'scripts/xray_train_direct.py' in x or 'scripts/xray_direct_queue.py' in x]
assert not active, '실행 중인 학습이 있습니다. 복원하지 마세요: '+str(active)
subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
subprocess.run(['uv','python','install','3.12.14'],check=True)
if not Path(PY).exists(): subprocess.run(['uv','venv',str(Path(PY).parent.parent),'--python','3.12.14'],check=True)
with (STORE/'environment_restore.log').open('a') as log:
    p=subprocess.run(['uv','pip','sync','--python',PY,str(STORE/'environment.lock.txt'),'--extra-index-url','https://download.pytorch.org/whl/cu126','--index-strategy','unsafe-best-match'],stdout=log,stderr=subprocess.STDOUT)
assert p.returncode==0, '환경 복원 로그 확인 필요'
freeze=subprocess.check_output(['uv','pip','freeze','--python',PY],text=True)
assert sorted(freeze.splitlines())==sorted((STORE/'environment.lock.txt').read_text().splitlines()), '환경 버전 불일치'
ROOT.mkdir(exist_ok=True)
for filename,expected in [
 ('xray_direct_20261006.zip','f477b58b769c238041cf1490b642428db4a56f31f9f4fc5b713469e6c2f983d2'),
 ('direct_support.zip','c14d41da54e82f050bf32d4e3b2ff783a590b5b556a64120ab50e6e1e61b3295')]:
    source=STORE/filename
    assert digest(source)==expected, '자료 묶음 해시 불일치: '+filename
    local=Path('/content')/filename; shutil.copy2(source,local)
    assert digest(local)==expected
    with zipfile.ZipFile(local) as z:
        assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
        z.extractall(ROOT)
    if filename=='xray_direct_20261006.zip':
        manifest=json.loads((ROOT/'BUNDLE_MANIFEST.json').read_text())
        assert all(digest(ROOT/n)==h for n,h in manifest.items())
assert digest(STORE/'rf-detr-small.pth')=='d81979a9213a2109345158ce9232668df4c1ae52e9b8db3f2ec0a8cbad959b33'
(ROOT/'weights').mkdir(exist_ok=True); shutil.copy2(STORE/'rf-detr-small.pth',ROOT/'weights/rf-detr-small.pth')
(STORE/'runs').mkdir(exist_ok=True)
if not (ROOT/'runs').exists(): (ROOT/'runs').symlink_to(STORE/'runs',target_is_directory=True)
assert (ROOT/'runs').resolve()==STORE/'runs'
print(subprocess.check_output([PY,'-c','import torch; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name())'],text=True))
print('DIRECT_RESTORE_VERIFIED',len(manifest),'files')


In [ ]:
# 기존 서버의 실행 종료를 확인한 뒤 사용. 학습 설정은 변경하지 않습니다.
import subprocess, time, json
from pathlib import Path
ROOT=Path('/content/xray_direct_project'); STORE=Path('/content/drive/MyDrive/xray_direct_training_20261006')
PY='/content/xray_direct_env312/bin/python'
assert (ROOT/'runs').resolve()==STORE/'runs'
active=[x for x in subprocess.check_output(['ps','-eo','pid,args'],text=True).splitlines() if 'scripts/xray_train_direct.py' in x or 'scripts/xray_direct_queue.py' in x]
assert not active, '이미 실행 중인 학습을 확인하세요: '+str(active)
# RecoveryStore가 manifest, 가중치, optimizer/scheduler, 난수 상태와 실행 설정을 검증합니다.
# 완료 run은 재학습하지 않고 선택 저장본 평가 단계로 넘어갑니다.
with (STORE/'queue_driver.log').open('a') as log:
    training_proc=subprocess.Popen([PY,'scripts/xray_direct_queue.py'],cwd=ROOT,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
(STORE/'queue_pid.json').write_text(json.dumps({'pid':training_proc.pid,'started_unix':time.time()}))
print('DIRECT_RF_TRAINING_STARTED',training_proc.pid,flush=True)
last=None
while training_proc.poll() is None:
    status={}
    for f in ['queue_rfdetr.json','runs/v2_direct20_rfdetr_cuda_20261006/progress.json']:
        if (STORE/f).exists():
            try: status[f]=json.loads((STORE/f).read_text())
            except ValueError: pass
    if status!=last: print(time.strftime('%Y-%m-%d %H:%M:%S UTC',time.gmtime()),status,flush=True); last=status
    time.sleep(30)
print('DIRECT_RF_QUEUE_EXIT',training_proc.returncode,flush=True)
print((STORE/'queue_rfdetr.json').read_text())
if training_proc.returncode:
    print((STORE/'queue_driver.log').read_text()[-6000:])
    for p in STORE.glob('v2_direct20_rfdetr_cuda_20261006_*.log'): print(p.name,p.read_text()[-5000:])


In [ ]:
# 읽기 전용 상태 확인. 학습·복원을 시작하지 않습니다.
import json, subprocess
from pathlib import Path
STORE=Path('/content/drive/MyDrive/xray_direct_training_20261006')
for name in ['queue_rfdetr.json','runs/v2_direct20_rfdetr_cuda_20261006/progress.json','runs/v2_direct20_rfdetr_cuda_20261006/execution.json']:
    p=STORE/name
    if p.exists(): print(name,p.read_text())
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.used,utilization.gpu','--format=csv'],text=True))
p=STORE/'v2_direct20_rfdetr_cuda_20261006_training.log'
if p.exists(): print(p.read_text()[-6000:])


In [ ]:
# 사용자 승인 CPU 런타임 복원: 같은 Drive 자료와 고정 환경
import hashlib, json, subprocess, sys, shutil, zipfile, os
os.environ['MPLBACKEND']='Agg'
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
STORE=Path('/content/drive/MyDrive/xray_direct_training_20261006')
ROOT=Path('/content/xray_direct_project')
PY='/content/xray_direct_env312/bin/python'
def digest(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)
    return h.hexdigest()
assert STORE.is_dir() and (STORE/'environment.lock.txt').is_file(), '기존 Drive 자료 연결 필요'
active=[x for x in subprocess.check_output(['ps','-eo','pid,args'],text=True).splitlines() if 'scripts/xray_train_direct.py' in x or 'scripts/xray_direct_queue.py' in x]
assert not active, '실행 중인 학습이 있습니다. 복원하지 마세요: '+str(active)
subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
subprocess.run(['uv','python','install','3.12.14'],check=True)
if not Path(PY).exists(): subprocess.run(['uv','venv',str(Path(PY).parent.parent),'--python','3.12.14'],check=True)
with (STORE/'environment_restore.log').open('a') as log:
    p=subprocess.run(['uv','pip','sync','--python',PY,str(STORE/'environment.lock.txt'),'--extra-index-url','https://download.pytorch.org/whl/cu126','--index-strategy','unsafe-best-match'],stdout=log,stderr=subprocess.STDOUT)
assert p.returncode==0, '환경 복원 로그 확인 필요'
freeze=subprocess.check_output(['uv','pip','freeze','--python',PY],text=True)
assert sorted(freeze.splitlines())==sorted((STORE/'environment.lock.txt').read_text().splitlines()), '환경 버전 불일치'
ROOT.mkdir(exist_ok=True)
for filename,expected in [
 ('xray_direct_20261006.zip','f477b58b769c238041cf1490b642428db4a56f31f9f4fc5b713469e6c2f983d2'),
 ('direct_support.zip','c14d41da54e82f050bf32d4e3b2ff783a590b5b556a64120ab50e6e1e61b3295')]:
    source=STORE/filename
    assert digest(source)==expected, '자료 묶음 해시 불일치: '+filename
    local=Path('/content')/filename; shutil.copy2(source,local)
    assert digest(local)==expected
    with zipfile.ZipFile(local) as z:
        assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
        z.extractall(ROOT)
    if filename=='xray_direct_20261006.zip':
        manifest=json.loads((ROOT/'BUNDLE_MANIFEST.json').read_text())
        assert all(digest(ROOT/n)==h for n,h in manifest.items())
assert digest(STORE/'rf-detr-small.pth')=='d81979a9213a2109345158ce9232668df4c1ae52e9b8db3f2ec0a8cbad959b33'
(ROOT/'weights').mkdir(exist_ok=True); shutil.copy2(STORE/'rf-detr-small.pth',ROOT/'weights/rf-detr-small.pth')
(STORE/'runs').mkdir(exist_ok=True)
if not (ROOT/'runs').exists(): (ROOT/'runs').symlink_to(STORE/'runs',target_is_directory=True)
assert (ROOT/'runs').resolve()==STORE/'runs'
print(subprocess.check_output([PY,'-c','import torch; print(torch.__version__, 'CPU', torch.get_num_threads())'],text=True))
print('DIRECT_RESTORE_VERIFIED',len(manifest),'files')


In [ ]:
# 사용자 승인 CPU 실행기를 저장하고 재개. 먼저 CPU 복원 셀 성공 확인.
import hashlib, subprocess, json, time
source='"""User-approved CUDA to CPU continuation; original source/binding stay intact."""\nimport argparse\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\nos.environ[\'MPLBACKEND\']=\'Agg\'\nimport xray_train_direct as training\nfrom xray_recovery import RecoveryStore, atomic_json, run_lock, sha256\nfrom xray_config import ROOT\n\nclass CPUContinuationStore(RecoveryStore):\n    def __init__(self, directory, config):\n        original=json.loads((Path(directory)/\'binding.json\').read_text())[\'config\']\n        assert config[\'device\']==\'cpu\' and original[\'device\']==\'cuda\'\n        # Only the explicitly approved execution device differs. All original\n        # data, package and source hashes still pass RecoveryStore\'s fingerprint.\n        checked=dict(config,device=\'cuda\')\n        super().__init__(directory,checked)\n        saved=self.latest()\n        if saved is None:raise RuntimeError(\'Verified full checkpoint required\')\n        atomic_json(Path(directory).parent/\'cpu_continuation.json\',dict(\n            approved_by=\'user 2026-10-07 CPU continuation request\',\n            original_device=\'cuda\',execution_device=\'cpu\',resume_epoch=saved[\'epoch\'],\n            original_fingerprint=self.fingerprint,launcher_sha256=sha256(Path(__file__)),\n            rng_policy=\'Restore CPU Python NumPy; CUDA RNG retained in previous checkpoint only\',\n            numeric_equivalence=False,started_unix=time.time()))\n        print(\'CPU_CONTINUATION_VERIFIED\',saved[\'epoch\'],self.fingerprint,flush=True)\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--approved-cpu-continuation\',action=\'store_true\',required=True);p.parse_args()\n    storage=(ROOT/\'runs\').resolve().parent\n    assert str(storage)==\'/content/drive/MyDrive/xray_direct_training_20261006\'\n    name=\'v2_direct20_rfdetr_cuda_20261006\'\n    state=storage/\'queue_rfdetr.json\'\n    training.RecoveryStore=CPUContinuationStore\n    original_restore=training.restore_rng\n    def restore_cpu(state,generator=None):\n        original_restore({k:v for k,v in state.items() if k not in (\'cuda\',\'mps\')},generator)\n    training.restore_rng=restore_cpu\n    a=argparse.Namespace(model=\'rfdetr\',data=\'data/xray_direct_training_20261006\',name=name,\n        epochs=20,batch=4,resolution=512,device=\'cpu\',yolo_weights=\'weights/yolo11s.pt\',\n        resume=True,epoch_validation=True,stop_after_epoch=0)\n    with run_lock(storage/\'queue_rfdetr\'):\n        try:\n            atomic_json(state,dict(status=\'running\',phase=\'training\',run=name,device=\'cpu\',updated_unix=time.time()))\n            with run_lock(ROOT/\'runs\'/name):training.run(a)\n            atomic_json(state,dict(status=\'running\',phase=\'test_evaluation\',run=name,device=\'cpu\',updated_unix=time.time()))\n            with (storage/(name+\'_test_evaluation.log\')).open(\'a\') as log:\n                subprocess.run([sys.executable,\'scripts/xray_evaluate_direct.py\',\'--run\',name,\'--device\',\'cpu\'],cwd=ROOT,stdout=log,stderr=subprocess.STDOUT,check=True)\n            atomic_json(state,dict(status=\'complete\',run=name,device=\'cpu\',test_evaluated=True,updated_unix=time.time()))\n        except BaseException as e:\n            atomic_json(state,dict(status=\'failed\',run=name,device=\'cpu\',error=repr(e),updated_unix=time.time()));raise\n\nif __name__==\'__main__\':main()\n'
assert hashlib.sha256(source.encode()).hexdigest()=='e47f7e933d038bd98f0a41887c0674eee86eca554384b4f618b7b3b3980f4eaa'
assert (ROOT/'runs').resolve()==STORE/'runs'
assert not [x for x in subprocess.check_output(['ps','-eo','pid,args'],text=True).splitlines() if 'scripts/xray_' in x]
p=ROOT/'scripts/xray_resume_direct_cpu.py';p.write_text(source)
(STORE/'cpu_continuation_launcher.py').write_text(source)
(STORE/'cpu_continuation_launcher.sha256').write_text(hashlib.sha256(source.encode()).hexdigest())
with (STORE/'v2_direct20_rfdetr_cuda_20261006_training.log').open('a') as log:
 proc=subprocess.Popen([PY,str(p),'--approved-cpu-continuation'],cwd=ROOT,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
(STORE/'queue_pid.json').write_text(json.dumps({'pid':proc.pid,'device':'cpu','started_unix':time.time()}))
print('CPU_RESUME_PROCESS_STARTED',proc.pid)
